XGBREGRESSION-(XGBOOST)

IMPORTS

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, root_mean_squared_error
from xgboost import XGBRegressor


In [2]:
df = pd.read_csv('cleaned_dataset.csv')
print(df.shape)
print(df.isna().sum())
print(df.duplicated().sum())
print(df['Calories'].describe())

(2532, 11)
StepTotal            0
TotalIntensity       0
AvgHeartRate         0
MaxHeartRate         0
MinHeartRate         0
SleepMinutes         0
WeightKg          1520
BMI               1520
Hour                 0
DayOfWeek            0
Calories             0
dtype: int64
0
count    2532.000000
mean      117.505529
std        75.949439
min        47.000000
25%        71.000000
50%        94.000000
75%       139.250000
max       933.000000
Name: Calories, dtype: float64


FEATURES & TARGET

In [3]:
X = df.drop(columns=['Calories'])
# HAN4EL EL COLUMNS MEN EL FEATURES 34AN TEB2A EL TARGET
y = df['Calories']
# CALORIES IS TARGET

X['HasWeight'] = X['WeightKg'].notna().astype(int)
X['HR_Range'] = X['MaxHeartRate'] - X['MinHeartRate']
X['Steps_per_Intensity'] = X['StepTotal'] / (X['TotalIntensity'] + 1)
X['Hour_sin'] = np.sin(2 * np.pi * X['Hour'] / 24)
X['Hour_cos'] = np.cos(2 * np.pi * X['Hour'] / 24)

ENCODING

In [4]:
# ONE HOT ENCODING FOR DAY OF THE WEEK
X_XGB = pd.get_dummies(X, columns=['DayOfWeek'], dtype=int)
print("ENCODING SHAPE = ", X_XGB.shape)
print("DAY OF THE WEEK COLUMNS", list(X_XGB.columns[14:]))
print("ALL COLUMNS", list(X_XGB.columns))
#CHANGED DAY OF WEEK TO SEPARATED DAYS

ENCODING SHAPE =  (2532, 21)
DAY OF THE WEEK COLUMNS ['DayOfWeek_Friday', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday']
ALL COLUMNS ['StepTotal', 'TotalIntensity', 'AvgHeartRate', 'MaxHeartRate', 'MinHeartRate', 'SleepMinutes', 'WeightKg', 'BMI', 'Hour', 'HasWeight', 'HR_Range', 'Steps_per_Intensity', 'Hour_sin', 'Hour_cos', 'DayOfWeek_Friday', 'DayOfWeek_Monday', 'DayOfWeek_Saturday', 'DayOfWeek_Sunday', 'DayOfWeek_Thursday', 'DayOfWeek_Tuesday', 'DayOfWeek_Wednesday']


TRAIN & TEST & GRID SEARCH

In [5]:
# MAKE TRAIN = 80%  MAKE TEST = 20%
X_train, X_test, y_train, y_test = train_test_split(
    X_XGB, y, test_size=0.2, random_state=0)

BEFORE GRID

In [6]:
base = XGBRegressor(random_state=0, n_jobs=-1)
base.fit(X_train, y_train)

print("=== Before GridSearchCV ===")
print(f"Train R2 : {r2_score(y_train, base.predict(X_train)):.4f}")
print(f"Test  R2 : {r2_score(y_test,  base.predict(X_test)):.4f}")
print(f"MSE      : {mean_squared_error(y_test, base.predict(X_test)):.2f}")
print(f"RMSE     : {root_mean_squared_error(y_test, base.predict(X_test)):.2f}")
print(f"MAE      : {mean_absolute_error(y_test, base.predict(X_test)):.2f}")

=== Before GridSearchCV ===
Train R2 : 0.9991
Test  R2 : 0.9343
MSE      : 406.66
RMSE     : 20.17
MAE      : 12.13


AFTER GRID

In [7]:
param_grid = {
    'n_estimators':     [200, 400],
    'learning_rate':    [0.03, 0.05, 0.1],
    'max_depth':        [3, 4, 5],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3],
    'reg_lambda':       [1, 5],
}

grid = GridSearchCV(
    XGBRegressor(random_state=0, n_jobs=-1),
    param_grid=param_grid,
    cv=5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    verbose=1
)
grid.fit(X_train, y_train)
best_xgb = grid.best_estimator_
print("Best Params:", grid.best_params_)
print("Best CV RMSE:", -grid.best_score_)

Fitting 5 folds for each of 288 candidates, totalling 1440 fits
Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 400, 'reg_lambda': 1, 'subsample': 0.8}
Best CV RMSE: 17.297664642333984


In [8]:
y_pred = best_xgb.predict(X_test)

print("=== After GridSearchCV ===")
print(f"Train R2 : {r2_score(y_train, best_xgb.predict(X_train)):.4f}")
print(f"Test  R2 : {r2_score(y_test, y_pred):.4f}")
print(f"MSE      : {mean_squared_error(y_test, y_pred):.2f}")
print(f"RMSE     : {root_mean_squared_error(y_test, y_pred):.2f}")
print(f"MAE      : {mean_absolute_error(y_test, y_pred):.2f}")

=== After GridSearchCV ===
Train R2 : 0.9928
Test  R2 : 0.9351
MSE      : 401.29
RMSE     : 20.03
MAE      : 11.18


BEST FEATURES

In [9]:
imp = pd.Series(best_xgb.feature_importances_, index=X_XGB.columns).sort_values(ascending=False)
print(imp.head(8).round(4))

TotalIntensity         0.4539
StepTotal              0.2432
AvgHeartRate           0.0694
WeightKg               0.0555
MaxHeartRate           0.0348
BMI                    0.0327
Steps_per_Intensity    0.0225
MinHeartRate           0.0151
dtype: float32


PREDICTION

In [10]:
print("Best Params:", grid.best_params_)
print("--------------")
print("R2 score")
print(f"{r2_score(y_test, y_pred):.4f}")
print("--------------")
print("MAE")
print(f"{mean_absolute_error(y_test, y_pred):.2f}")
print("--------------")
print("MSE")
print(f"{mean_squared_error(y_test, y_pred):.2f}")
print("--------------")
print("RMSE")
print(f"{root_mean_squared_error(y_test, y_pred):.2f}")
print("--------------")

Best Params: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 400, 'reg_lambda': 1, 'subsample': 0.8}
--------------
R2 score
0.9351
--------------
MAE
11.18
--------------
MSE
401.29
--------------
RMSE
20.03
--------------


In [11]:
# Sensitivity analysis (للتوضيح بس: الإعدادات اتضبطت على split 80/20)
best_params = grid.best_params_
print("=== XGBoost across test_sizes ===")
for size in [0.1, 0.2, 0.3, 0.4]:
    Xtr, Xte, ytr, yte = train_test_split(X_XGB, y, test_size=size, random_state=0)
    m = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    print(f"--- Test Size: {size*100:.0f}% ---")
    print(f"R2   : {r2_score(yte, p):.4f}")
    print(f"MAE  : {mean_absolute_error(yte, p):.2f}")
    print(f"MSE  : {mean_squared_error(yte, p):.2f}")
    print(f"RMSE : {root_mean_squared_error(yte, p):.2f}")

=== XGBoost across test_sizes ===


--- Test Size: 10% ---
R2   : 0.9238
MAE  : 12.58
MSE  : 526.15
RMSE : 22.94
--- Test Size: 20% ---
R2   : 0.9351
MAE  : 11.18
MSE  : 401.29
RMSE : 20.03
--- Test Size: 30% ---
R2   : 0.9288
MAE  : 11.15
MSE  : 405.11
RMSE : 20.13
--- Test Size: 40% ---
R2   : 0.9328
MAE  : 11.51
MSE  : 403.10
RMSE : 20.08


TESTTING MODEL WITHOUT SOME FEATURES

In [12]:
cols_no_user = [c for c in X_XGB.columns if c not in ['WeightKg', 'BMI', 'HasWeight']]
m = XGBRegressor(**grid.best_params_, random_state=0, n_jobs=-1)
m.fit(X_train[cols_no_user], y_train)
p = m.predict(X_test[cols_no_user])
print(f"R2 : {r2_score(y_test, p):.4f}")
print(f"RMSE: {root_mean_squared_error(y_test, p):.2f}")
print(f"MSE  : {mean_squared_error(y_test, p):.2f}")

R2 : 0.8966
RMSE: 25.29
MSE  : 639.76


TESTING MODEL ON UNSEEN DATA

In [13]:
cols = [c for c in X_XGB.columns if c not in ['WeightKg', 'BMI', 'HasWeight']]
has_w = X_XGB['HasWeight'] == 1

tests = {
    'A: train on weighted users -> test on OTHER people': (has_w, ~has_w),
    'B: train on other people -> test on weighted users': (~has_w, has_w),
}

for name, (tr, te) in tests.items():
    m = XGBRegressor(**grid.best_params_, random_state=0, n_jobs=-1)
    m.fit(X_XGB.loc[tr, cols], y[tr])
    p = m.predict(X_XGB.loc[te, cols])
    base_rmse = root_mean_squared_error(y[te], np.full(te.sum(), y[tr].mean()))
    print(name)
    print(f"  train rows={tr.sum()} | test rows={te.sum()}")
    print(f"  R2   : {r2_score(y[te], p):.4f}")
    print(f"  RMSE : {root_mean_squared_error(y[te], p):.2f}  (predict-the-mean RMSE: {base_rmse:.2f})")
    print(f"  MAE  : {mean_absolute_error(y[te], p):.2f}")

A: train on weighted users -> test on OTHER people
  train rows=1012 | test rows=1520
  R2   : 0.7459
  RMSE : 36.33  (predict-the-mean RMSE: 73.61)
  MAE  : 25.89
B: train on other people -> test on weighted users
  train rows=1520 | test rows=1012
  R2   : 0.7723
  RMSE : 38.45  (predict-the-mean RMSE: 81.96)
  MAE  : 24.49


THE MODEL IS PERFECT ON UNSEEN DATA